In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import faiss

### Step-1. Data Loading

In [2]:
# Ratings

ratings = pd.read_csv("ml-1m/ratings.dat", 
                    sep = "::",                     # The separator used to differentiate columns
                    engine="python",                # Using python engine as the default is c engine which does not support two character separators
                    names=["user_id", "movie_id" , "rating", "timestamp"],
                    )

In [3]:
# Users

users = pd.read_csv("ml-1m/users.dat",
                    sep = "::",
                    engine="python",
                    names=["user_id", "gender", "age", "occupation", "zip_code"]
                    )

In [4]:
# Movies

movies = pd.read_csv("ml-1m/movies.dat",
                     sep = "::",
                     engine = "python",
                     names = ["movie_id", "title", "genre"]
                     )

### Step-2. Data Exploration

In [5]:
print("Ratings: ", ratings.shape)
print(ratings.head())

Ratings:  (1000209, 4)
   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291


In [6]:
print("Users: ", users.shape)
print(users.head())

Users:  (6040, 5)
   user_id gender  age  occupation zip_code
0        1      F    1          10    48067
1        2      M   56          16    70072
2        3      M   25          15    55117
3        4      M   45           7    02460
4        5      M   25          20    55455


In [7]:
print("Movies: ", movies.shape)
print(movies.head())

Movies:  (3883, 3)
   movie_id                               title                         genre
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy


In [8]:
# Rating distribution

print("Rating Distribution: ")
ratings['rating'].value_counts().sort_index()

Rating Distribution: 


rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

In [9]:
# Unique Users

print("Unique Users: ")
ratings['user_id'].nunique()

Unique Users: 


6040

In [10]:
# Unique movies

print("Unique Movies that are given rating: ")
ratings['movie_id'].nunique()

Unique Movies that are given rating: 


3706

In [11]:
# User age values

print("User age values: ")
sorted(users["age"].unique())

User age values: 


[np.int64(1),
 np.int64(18),
 np.int64(25),
 np.int64(35),
 np.int64(45),
 np.int64(50),
 np.int64(56)]

In [12]:
# Occupation Values

print("Occupation values: ")
sorted(users["occupation"].unique())

Occupation values: 


[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20)]

In [13]:
ratings.describe()

,user_id,movie_id,rating,timestamp
count,1.000209e+06,1.000209e+06,1.000209e+06,1.000209e+06
mean,3.024512e+03,1.865540e+03,3.581564e+00,9.722437e+08
std,1.728413e+03,1.096041e+03,1.117102e+00,1.215256e+07
min,1.000000e+00,1.000000e+00,1.000000e+00,9.567039e+08
25%,1.506000e+03,1.030000e+03,3.000000e+00,9.653026e+08
50%,3.070000e+03,1.835000e+03,4.000000e+00,9.730180e+08
75%,4.476000e+03,2.770000e+03,4.000000e+00,9.752209e+08
max,6.040000e+03,3.952000e+03,5.000000e+00,1.046455e+09


In [14]:
# Unique movies

print("Unique Movies: ")
movies['movie_id'].nunique()

Unique Movies: 


3883

In [15]:
movies.describe()

,movie_id
count,3883.000000
mean,1986.049446
std,1146.778349
min,1.000000
25%,982.500000
50%,2010.000000
75%,2980.500000
max,3952.000000


In [16]:
users.describe()

,user_id,age,occupation
count,6040.000000,6040.000000,6040.000000
mean,3020.500000,30.639238,8.146854
std,1743.742145,12.895962,6.329511
min,1.000000,1.000000,0.000000
25%,1510.750000,25.000000,3.000000
50%,3020.500000,25.000000,7.000000
75%,4530.250000,35.000000,14.000000
max,6040.000000,56.000000,20.000000


### Step-3. Preprocessing

In [17]:
# We wanted to know whether the user liked the movie or not rather then the exact rating value he had given. Hence we convert ratings into positive interactions with ratings >= 4.

ratings["label"] = (ratings["rating"] >= 4).astype(int)     # Sets the label value to 1, if ratings is 4 or 5.
ratings.head()

,user_id,movie_id,rating,timestamp,label
0,1,1193,5,978300760,1
1,1,661,3,978302109,0
2,1,914,3,978301968,0
3,1,3408,4,978300275,1
4,1,2355,5,978824291,1


In [18]:
positive_interactions = ratings[ratings['label'] == 1].reset_index(drop=True)

In [19]:
print("Total interactions: ", ratings.shape[0])
print("Positive interactions: ", positive_interactions.shape[0])

Total interactions:  1000209
Positive interactions:  575281


In [20]:
# From data exploration, we find that the max. value of movie_id is 3952 but total unique movies are 3706. Hence we remap the ids.
# Also remapping to make the ids 0-indexed 

unique_movie_id = movies["movie_id"].unique()
movie_id_map = {mid : idx for idx, mid in enumerate(unique_movie_id)}
positive_interactions["movie_idx"] = positive_interactions["movie_id"].map(movie_id_map)
movies["movie_idx"] = movies["movie_id"].map(movie_id_map)
num_movies = len(unique_movie_id)
print("Total no. of movies: ",num_movies)

Total no. of movies:  3883


In [21]:
# Remapping the IDs of users such that they are 0-indexed

unique_user_id = users["user_id"].unique()
user_id_map = {mid : idx for idx, mid in enumerate(unique_user_id)}
positive_interactions["user_idx"] = positive_interactions["user_id"].map(user_id_map)
users["user_idx"] = users["user_id"].map(user_id_map)
num_users = len(unique_user_id)
print("Total no. of users: ",num_users)

Total no. of users:  6040


In [22]:
# We normalize the age so that larger age values such as 56, does not scale the input to MLP
# This also ensures the ordinal realtionship of the age

users["age_normalized"] = users["age"] / users["age"].max()

In [23]:
# Occupation is already categorical and we keep the same, going forward we convert them into embeddings to differentiate between them

num_occupations = users["occupation"].nunique()
print("Total no. of occupations: ", num_occupations)

Total no. of occupations:  21


In [24]:
# Encoding the movie genres (Each movie can have multiple genres)

all_genres = sorted(set(g for genre_list in movies["genre"].str.split("|") for g in genre_list))
genre_idx = {g : i for i,g in enumerate(all_genres)}
num_genres = len(all_genres)
print("Total no. of genres: ", num_genres)

Total no. of genres:  18


In [25]:
movies["genre_ids"] = movies["genre"].apply(
                                    lambda x : [genre_idx[g] for g in x.split("|")]
                                    )
movies.head()

,movie_id,title,genre,movie_idx,genre_ids
0,1,Toy Story (1995),Animation|Children's|Comedy,0,"[2, 3, 4]"
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1,"[1, 3, 8]"
2,3,Grumpier Old Men (1995),Comedy|Romance,2,"[4, 13]"
3,4,Waiting to Exhale (1995),Comedy|Drama,3,"[4, 7]"
4,5,Father of the Bride Part II (1995),Comedy,4,[4]


### Step-4. Train-Test Split

In [26]:
# Random Splitting can't be done here, it should usually be done based on the timestamp as the last interaction is dependent on the previous interactions.

positive_interactions = positive_interactions.sort_values(["user_idx", "timestamp"])

In [27]:
test_df = positive_interactions.groupby("user_idx").tail(1)
train_df = positive_interactions.drop(test_df.index)

In [28]:
print("Train size: ", train_df.shape[0])
print("Test size: ", test_df.shape[0])

Train size:  569243
Test size:  6038


### Step-5. Building PyTorch Dataset and DataLoader

In [29]:
# User feature lookups : occupation & age

user_occupation = users.set_index("user_idx")["occupation"].to_dict()
user_age = users.set_index("user_idx")["age_normalized"].to_dict()

In [30]:
# Movie feature lookups : genre_ids

movie_genres = movies.set_index("movie_idx")["genre_ids"].to_dict()

In [31]:
class InteractionDataset(Dataset):
    def __init__(self, df, user_occupation, user_age, movie_genres):
        self.user_idx = df["user_idx"].values
        self.movie_idx = df["movie_idx"].values
        self.user_occupation = user_occupation
        self.user_age = user_age
        self.movie_genres = movie_genres

    def __len__(self):
        return len(self.user_idx)

    def __getitem__(self, idx):
        u_idx = self.user_idx[idx]
        m_idx = self.movie_idx[idx]
        return {
            "user_id" : u_idx,
            "occupation" : self.user_occupation[u_idx],
            "age" : self.user_age[u_idx],
            "movie_id" : m_idx,
            "genres" : self.movie_genres[m_idx]         # This is a variable-length list
        }        

In [32]:
# To encode the variable length list of genres we use collate_fn

def collate_fn(batch):
    user_ids = torch.tensor([b["user_id"] for b in batch], dtype=torch.long)
    occupations = torch.tensor([b["occupation"] for b in batch], dtype=torch.long)
    ages = torch.tensor([b["age"] for b in batch], dtype=torch.float32)
    movie_ids = torch.tensor([b["movie_id"] for b in batch], dtype=torch.long)

    genre_list = []
    genre_offsets = [0]
    for b in batch:
        genre_list.extend(b["genres"])
        genre_offsets.append(genre_offsets[-1] + len(b["genres"]))
    genre_offsets = genre_offsets[:-1]

    return {
        "user_id" : user_ids,
        "occupation" : occupations,
        "age" : ages,
        "movie_id" : movie_ids,
        "genres_list" : torch.tensor(genre_list, dtype=torch.long),
        "genre_offsets" : torch.tensor(genre_offsets, dtype=torch.long)
    }

In [33]:
train_dataset = InteractionDataset(train_df, user_occupation, user_age, movie_genres)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, collate_fn=collate_fn)

In [34]:
# Sanity check 
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)

user_id torch.Size([512])
occupation torch.Size([512])
age torch.Size([512])
movie_id torch.Size([512])
genres_list torch.Size([1096])
genre_offsets torch.Size([512])


### Step-6. User Tower, Movie Tower, Two-tower model

In [35]:
class UserTower(nn.Module):
    def __init__(self, num_users, num_occupations, embed_dim=64):
        super().__init__()
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.occ_embed = nn.Embedding(num_occupations, 16)
        self.mlp = nn.Sequential(
                            nn.Linear(embed_dim + 16 + 1, 128),  # +1 for age
                            nn.ReLU(),
                            nn.Linear(128, 64)
                            )

    def forward(self, user_id, occupation, age):
        u = self.user_embed(user_id)
        o = self.occ_embed(occupation)
        x = torch.cat([u, o, age.unsqueeze(-1)], dim=-1)
        return self.mlp(x)

In [36]:
class MovieTower(nn.Module):
    def __init__(self, num_movies, num_genres, embed_dim = 64):
        super().__init__()
        self.movie_embed = nn.Embedding(num_movies, embed_dim)
        self.genre_embed = nn.EmbeddingBag(num_genres, 16, mode="mean")
        self.mlp = nn.Sequential(
                            nn.Linear(embed_dim + 16, 128),
                            nn.ReLU(),
                            nn.Linear(128, 64)
                            )

    def forward(self, movie_id, genres_list, genres_offsets):
        m = self.movie_embed(movie_id)
        g = self.genre_embed(genres_list, genres_offsets)
        x = torch.cat([m, g], dim=-1)
        return self.mlp(x)

In [37]:
class TwoTower(nn.Module):
    def __init__(self, user_tower, movie_tower):
        super().__init__()
        self.user_tower = user_tower
        self.movie_tower = movie_tower

    def forward(self, batch):
        user_emb = self.user_tower(batch["user_id"], batch["occupation"], batch["age"])
        movie_emb = self.movie_tower(batch["movie_id"], batch["genres_list"], batch["genre_offsets"])
        return user_emb, movie_emb

In [38]:
embed_dim = 64
user_tower = UserTower(num_users, num_occupations, embed_dim)
item_tower = MovieTower(num_movies, num_genres, embed_dim)
model = TwoTower(user_tower, item_tower)

batch = next(iter(train_loader))
user_emb, item_emb = model(batch)
print("User embedding shape:", user_emb.shape)
print("Item embedding shape:", item_emb.shape)

User embedding shape: torch.Size([512, 64])
Item embedding shape: torch.Size([512, 64])


### Step-7. Softmax Loss

In [39]:
def in_batch_sotmax_loss(user_emb, movie_emb, temperature = 0.05):
    user_emb = F.normalize(user_emb, dim=-1)
    movie_emb = F.normalize(movie_emb, dim=-1)

    logits = user_emb @ movie_emb.T / temperature
    labels = torch.arange(logits.size(0), device = logits.device)

    return F.cross_entropy(logits, labels)

### Step-8. Training Loop

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 50

In [41]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = {k : v.to(device) for k,v in batch.items()}

        user_emb, movie_emb = model(batch)
        loss = in_batch_sotmax_loss(user_emb, movie_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - loss : {avg_loss:.6f}")

Epoch 1/50 - loss : 6.216588
Epoch 2/50 - loss : 6.054073
Epoch 3/50 - loss : 5.929654
Epoch 4/50 - loss : 5.849250
Epoch 5/50 - loss : 5.799087
Epoch 6/50 - loss : 5.765079
Epoch 7/50 - loss : 5.738743
Epoch 8/50 - loss : 5.717262
Epoch 9/50 - loss : 5.699103
Epoch 10/50 - loss : 5.683131
Epoch 11/50 - loss : 5.668274
Epoch 12/50 - loss : 5.654824
Epoch 13/50 - loss : 5.642776
Epoch 14/50 - loss : 5.631948
Epoch 15/50 - loss : 5.621759
Epoch 16/50 - loss : 5.612344
Epoch 17/50 - loss : 5.603626
Epoch 18/50 - loss : 5.595368
Epoch 19/50 - loss : 5.587798
Epoch 20/50 - loss : 5.580505
Epoch 21/50 - loss : 5.573564
Epoch 22/50 - loss : 5.567078
Epoch 23/50 - loss : 5.561046
Epoch 24/50 - loss : 5.554775
Epoch 25/50 - loss : 5.549085
Epoch 26/50 - loss : 5.543158
Epoch 27/50 - loss : 5.537916
Epoch 28/50 - loss : 5.532449
Epoch 29/50 - loss : 5.527108
Epoch 30/50 - loss : 5.522060
Epoch 31/50 - loss : 5.516817
Epoch 32/50 - loss : 5.511998
Epoch 33/50 - loss : 5.507135
Epoch 34/50 - loss 

In [42]:
torch.save(model.state_dict(), "two_tower_model.pt")

### Step-8. Evaluation

In [43]:
model.load_state_dict(torch.load("two_tower_model.pt"))
model.to(device)
model.eval()

with torch.no_grad():
    all_movie_ids = torch.arange(num_movies, device=device)

    genre_list_all = []
    genre_offsets_all = [0]
    for idx in range(num_movies):
        genres = movie_genres[idx]
        genre_list_all.extend(genres)
        genre_offsets_all.append(genre_offsets_all[-1] + len(genres))

    genre_offsets_all = genre_offsets_all[:-1]

    genre_list_all = torch.tensor(genre_list_all, dtype=torch.long, device=device)
    genre_offsets_all = torch.tensor(genre_offsets_all, dtype=torch.long, device=device)

    all_movie_embeddings = model.movie_tower(all_movie_ids, genre_list_all, genre_offsets_all)
    all_movie_embeddings = F.normalize(all_movie_embeddings, dim=-1).cpu().numpy()

print("All movie embeddings shape: ", all_movie_embeddings.shape)

All movie embeddings shape:  (3883, 64)


In [44]:
index = faiss.IndexFlatIP(all_movie_embeddings.shape[1])
index.add(all_movie_embeddings.astype("float32"))
print("FAISS index size: ", index.ntotal)

FAISS index size:  3883


In [45]:
def evaluate(test_df, k=10):
    model.eval()
    recalls = []
    ndcgs = []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            u_idx = row["user_idx"]
            true_movie = row["movie_idx"]

            occ = torch.tensor([user_occupation[u_idx]], dtype=torch.long, device=device)
            age = torch.tensor([user_age[u_idx]], dtype=torch.float32, device=device)
            uid = torch.tensor([u_idx], dtype=torch.long, device=device)

            user_emb = user_tower(uid, occ, age)
            user_emb = F.normalize(user_emb, dim=-1).cpu().numpy().astype("float32")

            scores, indices = index.search(user_emb, k)
            retrieved = indices[0]

            if true_movie in retrieved:
                recalls.append(1)
                rank = np.where(retrieved == true_movie)[0][0]
                ndcgs.append(1 / np.log2(rank + 2))
            
            else:
                recalls.append(0)
                ndcgs.append(0)

    recall_at_k = np.mean(recalls)
    ndcgs_at_k = np.mean(ndcgs)
    return recall_at_k, ndcgs_at_k

In [46]:
recall, ndcg = evaluate(test_df, k=10)

print(f"Recall@10: {recall:.4f}")
print(f"NDCG@10: {ndcg:.4f}")

Recall@10: 0.0181
NDCG@10: 0.0078
